# Improved CNN Architecture

Building on baseline insights, we design a deeper CNN with regularization techniques.

### Architecture Design Rationale:

- **Deeper Network**: 3 convolutional layers to capture hierarchical features
- **Progressive Filters**: 32 → 64 → 128 filters for increasing feature complexity
- **Regularization**: Dropout layer to prevent overfitting
- **Efficient Training**: ReLU activation for fast convergence

In [ ]:
# Improved CNN architecture
import tensorflow as tf
from keras import layers, models

NUM_CLASSES = 28

model = models.Sequential([
    # First convolutional block
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(200, 200, 3)),
    layers.MaxPooling2D(2, 2),
    
    # Second convolutional block  
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    # Third convolutional block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    
    # Classification head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),  # Regularization
    layers.Dense(NUM_CLASSES, activation='softmax')
])

print("Improved Model Architecture:")
model.summary()

### Training Configuration

**Optimizer**: Adam (adaptive learning rate)  
**Loss Function**: Sparse categorical crossentropy  
**Callbacks**: Early stopping and model checkpointing

In [ ]:
# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2, 
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "initial_model", 
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train the model
print("Training improved CNN model...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    callbacks=callbacks,
    epochs=10,
    verbose=1
)

print("✓ Training complete")

### Model Evaluation

In [ ]:
# Visualize training progress
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation', linewidth=2)
plt.title('Model Accuracy', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation', linewidth=2)
plt.title('Model Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate model performance
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)

print("=" * 50)
print("IMPROVED MODEL PERFORMANCE")
print("=" * 50)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
print("=" * 50)

In [ ]:
# Generate predictions for detailed analysis
print("Generating predictions for validation set...")

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Classification report
print("\nDetailed Classification Report:")
print("=" * 80)
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix visualization
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(14, 14))
disp.plot(ax=ax, xticks_rotation=45, cmap='Blues', values_format='d')
plt.title("Confusion Matrix - ASL Alphabet Classifier", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()